In [5]:
import pandas as pd
RAW = "data/raw/"

customers   = pd.read_csv(RAW + "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
orders      = pd.read_csv(RAW + "olist_orders_dataset.csv")
items       = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments    = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
products    = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers     = pd.read_csv(RAW + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(RAW + "product_category_name_translation.csv")

tables = {"customers": customers, "geolocation": geolocation, "orders": orders,
          "items": items, "payments": payments, "reviews": reviews,
          "products": products, "sellers": sellers,
          "category_translation": category_translation}

In [6]:
for name, df in tables.items():
    print(f"\n===== {name} {df.shape} =====")
    miss = df.isnull().sum()
    print("Missing:\n", miss[miss > 0] if miss.sum() else "none")
    print("Fully duplicated rows:", df.duplicated().sum())


===== customers (99441, 5) =====
Missing:
 none
Fully duplicated rows: 0

===== geolocation (1000163, 5) =====
Missing:
 none
Fully duplicated rows: 261831

===== orders (99441, 8) =====
Missing:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Fully duplicated rows: 0

===== items (112650, 7) =====
Missing:
 none
Fully duplicated rows: 0

===== payments (103886, 5) =====
Missing:
 none
Fully duplicated rows: 0

===== reviews (99224, 7) =====
Missing:
 review_comment_title      87656
review_comment_message    58247
dtype: int64
Fully duplicated rows: 0

===== products (32951, 9) =====
Missing:
 product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Fully duplicated rows: 0

===== s

In [7]:
print("customers.customer_id unique:", customers.customer_id.is_unique)
print("customers.customer_unique_id unique:", customers.customer_unique_id.is_unique, "(repeat customers)")
print("orders.order_id unique:", orders.order_id.is_unique)
print("products.product_id unique:", products.product_id.is_unique)
print("sellers.seller_id unique:", sellers.seller_id.is_unique)
print("items (order_id, order_item_id) dup:", items.duplicated(["order_id", "order_item_id"]).sum())
print("payments (order_id, payment_sequential) dup:", payments.duplicated(["order_id", "payment_sequential"]).sum())
print("reviews.review_id duplicated:", reviews.review_id.duplicated().sum())
print("reviews.order_id duplicated:", reviews.order_id.duplicated().sum())

customers.customer_id unique: True
customers.customer_unique_id unique: False (repeat customers)
orders.order_id unique: True
products.product_id unique: True
sellers.seller_id unique: True
items (order_id, order_item_id) dup: 0
payments (order_id, payment_sequential) dup: 0
reviews.review_id duplicated: 814
reviews.order_id duplicated: 551


In [8]:
print(orders.order_status.value_counts(), "\n")
print(payments.payment_type.value_counts(), "\n")
print("Payments with installments = 0:", (payments.payment_installments == 0).sum())
print("Payments with value = 0:", (payments.payment_value == 0).sum())
print("Review scores:\n", reviews.review_score.value_counts().sort_index(), "\n")
print(items[["price", "freight_value"]].describe())
print("\nOrders date columns are:", orders.order_purchase_timestamp.dtype, "(should be datetime)")


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64 

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64 

Payments with installments = 0: 2
Payments with value = 0: 9
Review scores:
 review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64 

               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000

Orders date columns are: str (should be datetime)


In [9]:
for col in ["price", "freight_value"]:
    q1, q3 = items[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((items[col] < q1 - 1.5*iqr) | (items[col] > q3 + 1.5*iqr)).sum()
    print(f"{col}: {n} outliers ({n/len(items):.1%})")

price: 8427 outliers (7.5%)
freight_value: 12134 outliers (10.8%)


## Data Quality Findings

- **Missing values:** `orders` has missing delivery dates (X rows), mostly orders that were cancelled or not yet delivered, so these are expected and will be kept as null.
- **Missing categories:** `products` has X rows with no category name and some missing size/weight values.
- **Duplicates:** `reviews` has duplicate `review_id` / `order_id` values (X rows). `geolocation` has about 261,000 duplicate rows.
- **Invalid values:** `payments` has X rows with value 0 and X with type `not_defined`.
- **Data types:** date columns are stored as text and must be converted to datetime.
- **Outliers:** `price` and `freight_value` have X% outliers by the IQR rule. These are genuine expensive items, so they are kept.